# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [ ]:
import asyncio
import json
import os
import time 
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

#client = AsyncOpenAI()
client = AsyncOpenAI(
    base_url="https://openai.vocareum.com/v1",
)


MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [ ]:
DATA_DIR = Path('data')   

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[5])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j06', 'snippet': 'Pied Piper Inc. has an opening for a Senior ML Engineer. Compression algorithms background a plus. We typically look for 7+ years in production ML and distributed systems, but exceptional candidates with less are welcome.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [ ]:
#Zero‑Shot
def prompt_zero_shot(snippet_text: str) -> list[dict]:
    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""
    return [
        {
            "role": "user",
            "content": (
                "Extract the following fields :\n"
                "- company\n"
                "- role\n"
                "- years_experience_required\n\n"
                "Return ONLY valid JSON with exactly these three keys.\n\n"
                f"Job snippet:\n{snippet_text}"
            )
        }        
    ]

    
#Few‑Shot
def prompt_few_shot(snippet_text: str) -> list[dict]:
    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
    examples = [
        {
            "role": "user",
            "content": "Job snippet:\nAcme Corp seeks a Data Analyst with 3 years experience."
        },
        {
            "role": "assistant",
            "content": '{"company": "Acme Corp", "role": "Data Analyst", "years_experience_required": 3}'
        },
        {
            "role": "user",
            "content": "Job snippet:\nGlobex LLC is hiring a Senior Backend Engineer. 5+ years required."
        },
        {
            "role": "assistant",
            "content": '{"company": "Globex LLC", "role": "Senior Backend Engineer", "years_experience_required": 5}'
        }
    ]

    final_user_msg = {
        "role": "user",
        "content": (
            "Extract company, role, and years_experience_required.\n"
            "Return ONLY valid JSON.\n\n"
            f"Job snippet:\n{snippet_text}"
        )
    }

    return examples + [final_user_msg]

#Structured 

def prompt_structured(snippet_text: str) -> list[dict]:
    """Strategy 3 — structured / role-based. Use a system prompt with persona + schema."""
    return [
        {
            "role": "system",
            "content": (
                "You are an expert recruiter. Extract structured information from job postings.\n"
                "Output ONLY valid JSON with this exact schema:\n"
                "{\n"
                '  "company": string,\n'
                '  "role": string,\n'
                '  "years_experience_required": number\n'
                "}\n"
                "Do not add extra fields. Do not explain."
            )
        },
        {
            "role": "user",
            "content": f"Job snippet:\n{snippet_text}"
        }
    ]

#Chain‑of‑Thought

def prompt_cot(snippet_text: str) -> list[dict]:
    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
    return [
        {
            "role": "user",
            "content": (
                "Extract company, role, and years_experience_required from this job posting.\n"
                "Identify the company, role, and required experience.\n"
                "After reasoning, output ONLY valid JSON with exactly these keys.\n\n"
                f"Job snippet:\n{snippet_text}"
            )
        }
    ]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}
 

test_snippet = snippets[5]["snippet"]

print("ZERO-SHOT:\n", prompt_zero_shot(test_snippet), "\n")
print("FEW-SHOT:\n", prompt_few_shot(test_snippet), "\n")
print("STRUCTURED:\n", prompt_structured(test_snippet), "\n")
print("CHAIN OF THOUGHT:\n", prompt_cot(test_snippet), "\n")




ZERO-SHOT:
 [{'role': 'user', 'content': 'Extract the following fields :\n- company\n- role\n- years_experience_required\n\nReturn ONLY valid JSON with exactly these three keys.\n\nJob snippet:\nPied Piper Inc. has an opening for a Senior ML Engineer. Compression algorithms background a plus. We typically look for 7+ years in production ML and distributed systems, but exceptional candidates with less are welcome.'}] 

FEW-SHOT:
 [{'role': 'user', 'content': 'Job snippet:\nAcme Corp seeks a Data Analyst with 3 years experience.'}, {'role': 'assistant', 'content': '{"company": "Acme Corp", "role": "Data Analyst", "years_experience_required": 3}'}, {'role': 'user', 'content': 'Job snippet:\nGlobex LLC is hiring a Senior Backend Engineer. 5+ years required.'}, {'role': 'assistant', 'content': '{"company": "Globex LLC", "role": "Senior Backend Engineer", "years_experience_required": 5}'}, {'role': 'user', 'content': 'Extract company, role, and years_experience_required.\nReturn ONLY valid J

## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [ ]:
import json
import re
import asyncio

 
# PARSE RESPONSE
 
def parse_response(text: str) -> dict | None:
    if text is None:
        return None
    cleaned = re.sub(r"```json|```", "", text).strip()
    try:
        return json.loads(cleaned)
    except:
        return None
 
# RUN ONE CALL
 
async def run_one(strategy_name: str, snippet: dict) -> dict:
    prompt_fn = STRATEGIES[strategy_name]
    messages = prompt_fn(snippet["snippet"])

    start = time.time()
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )
    latency = time.time() - start

    raw_text = response.choices[0].message.content
    parsed = parse_response(raw_text)

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_tokens = input_tokens + output_tokens
    cost = total_tokens / 1_000_000

    return {
        "strategy": strategy_name,
        "snippet_id": snippet["id"],
        "raw": raw_text,
        "parsed": parsed,
        "cost": cost,
        "latency": latency,
    }

# RUN ALL (40 CALLS)
 
async def run_all() -> list[dict]:
    tasks = []
    for strategy_name in STRATEGIES.keys():
        for snippet in snippets:
            tasks.append(run_one(strategy_name, snippet))
    return await asyncio.gather(*tasks)
 
# RUN ONE STRATEGY ONLY
 
async def run_one_strategy(strategy_name: str) -> list[dict]:
    tasks = [run_one(strategy_name, snippet) for snippet in snippets]
    return await asyncio.gather(*tasks)

# zero_shot_results = await run_one_strategy("zero_shot")
# zero_shot_results[0]

# few_shot_results = await run_one_strategy("few_shot")
# few_shot_results[0]

# structured_results = await run_one_strategy("structured")
# structured_results[0]

# cot_results = await run_one_strategy("cot")
# cot_results[0]


In [26]:
# Run all


results = await run_all()
print(f"Got {len(results)} results.")
results[11]

#1–10 - zero_shot
#11–20 - few_shot
#21–30 - structured
#31–40 - cot


Got 40 results.


{'strategy': 'few_shot',
 'snippet_id': 'j02',
 'raw': '```json\n{\n  "company": "Northwind Ltd",\n  "role": "Data Analyst",\n  "years_experience_required": 2\n}\n```',
 'parsed': {'company': 'Northwind Ltd',
  'role': 'Data Analyst',
  'years_experience_required': 2},
 'cost': 0.000195,
 'latency': 1.389007329940796}

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [ ]:
import json
import re
import time
import asyncio 
 
MODEL = "gpt-4o"          # or your IITM model name
TEMPERATURE = 0.0

# STRATEGIES dict (you already have these prompt fns)
STRATEGIES = {
    "zero_shot": prompt_zero_shot,
    "few_shot": prompt_few_shot,
    "structured": prompt_structured,
    "cot": prompt_cot,
}

 
# DATA LOADING
 
def load_jsonl(path: str) -> list[dict]:
    with open(path, "r") as f:
        return [json.loads(line) for line in f]

snippets = load_jsonl("data/job_snippets.jsonl")
golden   = load_jsonl("data/golden_set.jsonl")

gold = {item["id"]: item for item in golden}
 
# PARSING
 
def parse_response(text: str) -> dict | None:
    if text is None:
        return None
    cleaned = re.sub(r"```json|```", "", text).strip()
    try:
        return json.loads(cleaned)
    except:
        return None

 
# STEP 3 — RUN ONE CALL
 
async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return dict with all captured fields."""
    prompt_fn = STRATEGIES[strategy_name]
    messages = prompt_fn(snippet["snippet"])

    start = time.time()
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )
    latency = time.time() - start

    raw_text = response.choices[0].message.content
    parsed = parse_response(raw_text)

    input_tokens = response.usage.prompt_tokens
    output_tokens = response.usage.completion_tokens
    total_tokens = input_tokens + output_tokens
    cost = total_tokens / 1_000_000 

    return {
        "strategy": strategy_name,
        "snippet_id": snippet["id"],
        "raw": raw_text,
        "parsed": parsed,
        "cost": cost,
        "latency": latency,
    }
 
# STEP 3 — RUN ALL (10 × 4 = 40)
 
async def run_all() -> list[dict]:
    tasks = []
    for strategy_name in STRATEGIES.keys():
        for snippet in snippets:
            tasks.append(run_one(strategy_name, snippet))
    results = await asyncio.gather(*tasks)
    return results

# one strategy only

async def run_one_strategy(strategy_name: str) -> list[dict]:
    tasks = [run_one(strategy_name, snippet) for snippet in snippets]
    return await asyncio.gather(*tasks)

 
# STEP 4 — SCORING FUNCTIONS
 
def score_accuracy(extracted: dict | None, gold_item: dict) -> int:
    
    if extracted is None:
        return 0
    score = 0
    for key in ["company", "role", "years_experience_required"]:
        if key in extracted and key in gold_item:
            if str(extracted[key]).strip().lower() == str(gold_item[key]).strip().lower():
                score += 1
    return score

def parse_success(extracted: dict | None) -> bool:
    return extracted is not None

async def score_llm_judge(snippet_text: str, extracted: dict | None, gold_item: dict) -> int:
    """Use gpt-4o as judge. """
    if extracted is None:
        return 1

    messages = [
        {"role": "system", "content": "Score from 1 to 4 using the rubric."},
        {"role": "user", "content": f"""
Snippet: {snippet_text}

Gold: {gold_item}
Extracted: {extracted}

Rubric:
4 – all three fields correct
3 – two correct, no fabricated data
2 – one correct OR fabricated a field
1 – none correct or unparsable
"""}
    ]

    resp = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0,
    )

    try:
        return int(resp.choices[0].message.content.strip())
    except:
        return 1


# STEP 4 — APPLY SCORING TO ALL 40 RESULTS

async def score_all(results: list[dict]) -> list[dict]:
    scored = []
    # build snippet text lookup
    snippet_text = {s["id"]: s["snippet"] for s in snippets}

    for r in results:
        gold_item = gold[r["snippet_id"]]
        acc   = score_accuracy(r["parsed"], gold_item)
        ps    = parse_success(r["parsed"])
        judge = await score_llm_judge(snippet_text[r["snippet_id"]], r["parsed"], gold_item)

        scored.append({
            **r,
            "accuracy": acc,
            "parse_success": ps,
            "llm_judge_score": judge,
        })

    return scored

 
# RUN All 
 
results = await run_all()
print(f"Got {len(results)} results.")

# Step 4: score against golden set
scored_results = await score_all(results)
print(f"Scored {len(scored_results)} results.")

# Inspect any one
scored_results[0]


Got 40 results.
Scored 40 results.


{'strategy': 'zero_shot',
 'snippet_id': 'j01',
 'raw': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": "5+"\n}\n```',
 'parsed': {'company': 'Acme Corp',
  'role': 'Senior Software Engineer',
  'years_experience_required': '5+'},
 'cost': 0.000108,
 'latency': 1.4489991664886475,
 'accuracy': 2,
 'parse_success': True,
 'llm_judge_score': 1}

In [32]:
# Apply scoring to all 40 results
# TODO: loop through results, attach accuracy + parse_success + llm_judge_score to each row
scored = []   # list of result dicts with scoring fields added
print(f'Scored {len(scored_results)} results.')

 

Scored 40 results.


## Step 5 — Build the comparison table

In [ ]:

import pandas as pd

df = pd.DataFrame(scored_results)

df = df.rename(columns={
    "cost": "cost_usd",
    "latency": "latency_s"
})

# summary table
summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency_s': 'median',
}).round(3)

summary.columns = [
    'Accuracy (mean of 3)',
    'Parse rate',
    'Judge score',
    'Total cost ($)',
    'Latency p50 (s)'
]

summary


,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,2.2,0.8,1.0,0.001,1.361
few_shot,2.9,1.0,1.0,0.002,1.220
structured,2.8,1.0,1.0,0.001,1.344
zero_shot,2.7,1.0,1.0,0.001,1.418


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```